# Strengthening authentication: length/complexity, randomness, MFA  (MODEL SOLUTION, instructor copy, do not distribute)

# Coding problem -- 1.2.c: TOTP Verifier

**Skill:** Mitigate Risk · **Maps to:** 1.2.C · **Language:** Python 3 (pytest)

## Problem
Implement the server-side Time-based One-Time Password (TOTP) verifier that an authentication
backend would call to check a user's second factor. Given the shared secret, a submitted code,
and the current time, return True if the code is valid for the current 30-second window.

This is the algorithm defined in RFC 6238 (TOTP) / RFC 4226 (HOTP), implemented using only
Python's standard library `hmac` and `hashlib` modules.

## Function contract
Implement `solve(secret: str, code: str, t: int) -> bool` in the code editor
(see the Reference section below).

## Input
- `secret` (`str`) -- the shared secret key, encoded as an **even-length lowercase hex string**
  (e.g. `"0a1b2c3d"` represents 4 bytes: `b'\x0a\x1b\x2c\x3d'`).
- `code` (`str`) -- the 6-digit code submitted by the user, zero-padded if necessary
  (e.g. `"007342"` or `"123456"`).
- `t` (`int`) -- the current Unix timestamp (seconds since epoch) at the moment of verification.

## Output
`True` if the computed TOTP code for the current 30-second step matches `code`; `False` otherwise.

## Algorithm (exact steps, deterministic)
1. Decode the secret: `key = bytes.fromhex(secret)`
2. Compute the time step: `step = t // 30`
3. Encode the step as an 8-byte big-endian integer: `msg = step.to_bytes(8, 'big')`
4. Compute HMAC-SHA1: `digest = hmac.new(key, msg, hashlib.sha1).digest()`
   (This produces a 20-byte bytes object.)
5. Dynamic truncation:
   a. `offset = digest[19] & 0x0f`
   b. `p = ((digest[offset] & 0x7f) << 24) | (digest[offset+1] << 16) | (digest[offset+2] << 8) | digest[offset+3]`
6. Compute the OTP: `otp = p % 1_000_000`
7. Zero-pad: `expected = str(otp).zfill(6)`
8. Return `code == expected`

## Examples
```python
# Using key=bytes.fromhex("3132333435363738393031323334353637383930") (RFC 4226 test vector)
# At t=59 (step=1): expected code is "287082"
solve("3132333435363738393031323334353637383930", "287082", 59)   -> True
solve("3132333435363738393031323334353637383930", "287082", 60)   -> False  # step changed
solve("3132333435363738393031323334353637383930", "000000", 59)   -> False  # wrong code
```

## Constraints
- Deterministic, offline, stdlib only (`hmac`, `hashlib`).
- Accept only the CURRENT step (no window drift / no adjacent steps).
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(secret: str, code: str, t: int) -> bool
```

### Input
- `secret` (`str`) -- even-length lowercase hex string encoding the TOTP shared secret key
  (e.g. `"0a1b2c3d"` = 4 bytes). Tests only use valid hex strings of 4-20 bytes.
- `code` (`str`) -- 6-character string of decimal digits, zero-padded (e.g. `"007342"`).
- `t` (`int`) -- current Unix timestamp (non-negative integer).

### Output
`bool` -- `True` if `code` matches the TOTP for the 30-second step containing `t`; else `False`.

### Algorithm (exact, deterministic -- all tests derive expected values from this spec)

```
key    = bytes.fromhex(secret)
step   = t // 30
msg    = step.to_bytes(8, byteorder='big')
digest = hmac.new(key, msg, hashlib.sha1).digest()   # 20 bytes

offset = digest[19] & 0x0f
p      = ((digest[offset] & 0x7f) << 24
         | digest[offset+1] << 16
         | digest[offset+2] << 8
         | digest[offset+3])
otp    = p % 1_000_000
expected = str(otp).zfill(6)

return code == expected
```

### RFC 4226 test vector (for implementation validation)
Secret (ASCII "12345678901234567890" = 20 bytes, hex below):
```
secret = "3132333435363738393031323334353637383930"
```
Known correct codes per HOTP counter (step for TOTP = t // 30):

| step (t value) | expected code |
|----------------|---------------|
| 1  (t = 30-59)  | 287082        |
| 2  (t = 60-89)  | 359152        |
| 3  (t = 90-119) | 969429        |

These are derived from the RFC 4226 test vectors. the tests uses them to validate any
implementation.

### Guarantees / rules
- Pure function, deterministic, no I/O, no network.
- Standard library only: `import hmac`, `import hashlib`.
- Only the CURRENT step is accepted (no window tolerance / no drift allowance).
- `code` is always exactly 6 characters. A code with fewer than 6 digits is already zero-padded
  by the caller.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Reference solution
"""Reference solution for 1.2.c. MUST pass 100% of the tests.
Also the default 'blackbox' implementation students can swap out (see the hidden grading tests)."""
import hmac
import hashlib


def solve(secret: str, code: str, t: int) -> bool:
    """Verify a 6-digit TOTP code against the shared secret at Unix timestamp t.

    Args:
        secret: even-length hex string encoding the shared key (e.g. "0a1b2c3d").
        code:   6-character zero-padded decimal string (e.g. "007342").
        t:      current Unix timestamp (int, seconds since epoch).
    Returns:
        True if code matches TOTP(secret, t // 30); False otherwise.
    """
    key = bytes.fromhex(secret)
    step = t // 30
    msg = step.to_bytes(8, byteorder='big')
    digest = hmac.new(key, msg, hashlib.sha1).digest()

    # RFC 4226 dynamic truncation
    offset = digest[19] & 0x0f
    p = (((digest[offset] & 0x7f) << 24)
         | (digest[offset + 1] << 16)
         | (digest[offset + 2] << 8)
         | digest[offset + 3])

    otp = p % 1_000_000
    expected = str(otp).zfill(6)
    return code == expected


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

RFC_SECRET = "3132333435363738393031323334353637383930"


# ---- visible cases (RFC 4226 test vectors) ----------------------------------

def test_rfc_step1_correct():
    """RFC 4226 step=1: t=59 -> code '287082'."""
    assert impl.solve(RFC_SECRET, "287082", 59) is True


def test_rfc_step2_correct():
    """RFC 4226 step=2: t=89 -> code '359152'."""
    assert impl.solve(RFC_SECRET, "359152", 89) is True


def test_rfc_step3_correct():
    """RFC 4226 step=3: t=90 -> code '969429'."""
    assert impl.solve(RFC_SECRET, "969429", 90) is True


def test_wrong_code_rejected():
    """A wrong code for a valid step must return False."""
    assert impl.solve(RFC_SECRET, "000000", 59) is False


def test_expired_code_rejected():
    """Code for step=1 must be rejected at t=60 (step=2)."""
    # step=1 code is "287082"; at t=60 step=2 expects "359152"
    assert impl.solve(RFC_SECRET, "287082", 60) is False


# ---- hidden / edge cases ----------------------------------------------------

def test_same_step_different_timestamp():
    """t=30 and t=59 share step=1; both should accept the same code."""
    assert impl.solve(RFC_SECRET, "287082", 30) is True
    assert impl.solve(RFC_SECRET, "287082", 59) is True


def test_step_boundary():
    """t=60 is exactly the start of step=2; must accept step=2 code only."""
    assert impl.solve(RFC_SECRET, "359152", 60) is True
    assert impl.solve(RFC_SECRET, "287082", 60) is False  # step=1 code rejected at step=2


def test_t_zero():
    """t=0 is step=0; computed value must be deterministic."""
    # Compute expected: step=0, verify True/False is consistent
    result_true = impl.solve("0a1b2c3d", "497680", 0)
    result_false = impl.solve("0a1b2c3d", "000000", 0)
    assert result_true is True
    assert result_false is False


def test_different_key_same_time():
    """Different keys produce different codes at the same timestamp."""
    # Both at t=0 (step=0)
    # key "0a1b2c3d" -> "497680"; key "deadbeef" -> "057490"
    assert impl.solve("0a1b2c3d", "497680", 0) is True
    assert impl.solve("deadbeef", "057490", 0) is True
    # Cross-key: "0a1b2c3d" code must fail for "deadbeef" key
    assert impl.solve("deadbeef", "497680", 0) is False


def test_replay_blocked():
    """Code valid at t=59 (step=1) must be rejected at t=60 (step=2)."""
    # This is the core replay-prevention property
    valid_code = "287082"  # step=1
    assert impl.solve(RFC_SECRET, valid_code, 59) is True
    assert impl.solve(RFC_SECRET, valid_code, 60) is False
    assert impl.solve(RFC_SECRET, valid_code, 119) is False  # step=3 also wrong


def test_code_must_be_exact_string():
    """Code matching is an exact string comparison after zero-padding."""
    # "287082" passes; "28708" (5 digits) or "2870820" (7) should not match
    assert impl.solve(RFC_SECRET, "28708", 59) is False
    assert impl.solve(RFC_SECRET, " 287082", 59) is False


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
